# Nettoyage et consolidation des données

Ce notebook assemble les trois sources collectées (LCSQA, Météo-France, IREP),réalise les jointures spatiales et temporelles, construit les features et la variable cible, analyse les valeurs manquantes et sauvegarde le DataFrame consolidé final sur S3.

## Pipeline

1. Chargement des données brutes depuis S3
2. Jointure spatiale entre LCSQA et  Météo-France (station la plus proche)
3. Jointure spatiale entre LCSQA et IREP (densité d'émissions dans un rayon de 5 km)
4. Construction des features temporelles et des lags PM2.5
5. Construction de la variable cible (dépassement seuil 25 µg/m³ / 24h)
6. Analyse des valeurs manquantes
7. Nettoyage et sélection des colonnes
8. Sauvegarde sur S3

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data_loader import load_lcsqa, clean_valeurs_negatives, load_meteo, load_irep, verifier_s3
from src.spatial     import join_meteo_to_lcsqa, compute_irep_density
from src.features    import add_temporal_features, add_lags, add_target

# 1. chargement des données brutes 

In [ ]:
df_lcsqa = load_lcsqa()
df_lcsqa = clean_valeurs_negatives(df_lcsqa)
print(f"LCSQA  : {df_lcsqa.shape[0]:,} lignes | {df_lcsqa['code_station'].nunique()} stations")
print(f"Période: {df_lcsqa['datetime_debut'].min()} → {df_lcsqa['datetime_debut'].max()}")

In [ ]:
df_meteo = load_meteo()
print(f"Météo  : {df_meteo.shape[0]:,} lignes | {df_meteo['code_station_meteo'].nunique()} stations")
print(f"Période: {df_meteo['datetime_meteo'].min()} → {df_meteo['datetime_meteo'].max()}")

In [ ]:
df_irep = load_irep()
print(f"IREP   : {df_irep.shape[0]:,} lignes | {df_irep['identifiant'].nunique()} établissements")
print(f"Années : {sorted(df_irep['annee'].unique())}")

## 2. Jointures spatiales

### 2.1 LCSQA - Météo-France

Pour chaque station LCSQA, on associe la station météo active la plus proche via la distance Haversine. Le merge est ensuite temporel exact sur l'heure.

Les stations météo inactives (moins de 1000 mesures de température) sont exclues du mapping pour éviter des associations avec des stations qui peuvent être fermées ou ouvertes tard et donc engendrera beacoup de valeurs manquantes

In [ ]:
df = join_meteo_to_lcsqa(df_lcsqa, df_meteo)
print(f"\nAprès jointure météo : {df.shape[0]:,} lignes")
print(f"Taux remplissage température : {df['temperature_c'].notna().mean():.1%}")
print(f"\nTaux par station :")
print(df.groupby('code_station')['temperature_c'].apply(
    lambda x: x.notna().mean()
).round(3).to_string())

### 2.2 LCSQA - IREP

Pour chaque station LCSQA et chaque année, on calcule le nombre d'installations industrielles comme étant un proxi de la somme des émissions PM des installations industrielles dans un rayon de 5 km.

Le taux de remplissage obtenu est 78,2% ce qui est cohérent vu que ces données sont dispoibles jusqu'en 2024 alors que notre base va jusqu'en 2025

In [ ]:
df = compute_irep_density(df, df_irep)
print(f"\nAprès jointure IREP : {df.shape[0]:,} lignes")
print(f"Taux remplissage IREP : {df['densite_emission_pm_kg'].notna().mean():.1%}")

## 3. Construction des features

Ici nous construisons les features qui seront utiles pour les anlyses et les modeles notemment les lag de concentrations de PM2.5 et les moyenne glissantes

In [ ]:
df = add_temporal_features(df)
df = add_lags(df)
df = add_target(df)

print(f"Shape après features : {df.shape}")
print(f"\nTaux de dépassement : {df['depasse_seuil_24h'].mean():.1%}")
print(f"Distribution cible :")
print(df['depasse_seuil_24h'].value_counts(dropna=False))

## 4. Analyse des valeurs manquantes

In [ ]:
# Taux de valeurs manquantes par colonne
na_pct = df.isna().mean().sort_values(ascending=False)
na_pct = na_pct[na_pct > 0]

print("Colonnes avec valeurs manquantes :")
print((na_pct * 100).round(1).to_string())

# Visualisation
fig, ax = plt.subplots(figsize=(10, 5))
na_pct.plot(kind="bar", ax=ax)
ax.set_title("Taux de valeurs manquantes par colonne")
ax.set_ylabel("Taux (%)")
ax.set_xlabel("")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 5. Nettoyage

### 5.1 Suppression des colonnes inutiles pour la modélisation

Les colonnes suivantes sont supprimées car elles ne sont pas des features pour le modèle, ce sont des métadonnées administratives ou des colonnes redondantes.

In [ ]:
cols_a_supprimer = [
    "datetime_fin",
    "organisme",
    "code_zas",
    "zas",
    "polluant",
    "type_d_influence",
    "reglementaire",
    "type_d_evaluation",
    "type_de_valeur",
    "unite_de_mesure",
    "taux_de_saisie",           # 100% NaN
    "couverture_temporelle",    # 100% NaN
    "couverture_de_donnees",    # 100% NaN
    "code_qualite",
    "validite",
    "pm25_valide",              # on utilise pm25_brute
    "pm25_max_24h",             # variable intermédiaire
    "code_station_meteo",       # identifiant technique
    "densite_emission_pm_kg",   # émissions non quantifiables
]

cols_presentes = [c for c in cols_a_supprimer if c in df.columns]
df = df.drop(columns=cols_presentes)
print(f"Colonnes supprimées : {len(cols_presentes)}")
print(f"Shape après suppression : {df.shape}")
print(f"\nColonnes restantes :")
print(df.columns.tolist())

Nous imputons les valeurs manquantes des variables temporelles en utilisant une interpolation linéaire par fenêtre de 6 au vu de la taille des données afin d'éviter de long temps de calculs. Les valeurs manquantes restantes (Pas de valeurs dans une fenetre de 6 ) sont simplement imputées par la médiane de la station concernée. le pourecntage de ces valeurs residuels etant tres faible, cette imputation n'a pas une grande influence.

In [ ]:
from src.features import imputer_par_fenetre

cols_a_imputer = [
    "pm25_brute",
    "temperature_c",
    "vent_vitesse_ms",
    "vent_direction_deg",
    "humidite_pct",
    "pluie_mm",
    #"nb_installations_5km",
]

df = imputer_par_fenetre(df, cols_a_imputer, fenetre=6)

print("NaN après imputation :")
print((df[cols_a_imputer].isna().mean() * 100).round(1).to_string())

In [ ]:
# Imputation par la médiane par station pour les NaN résiduels
cols_mediane = [
    "pm25_brute",
    "temperature_c", 
    "vent_vitesse_ms",
    "vent_direction_deg",
    "humidite_pct",
    "pluie_mm",
]

for col in cols_mediane:
    df[col] = df.groupby("code_station")[col].transform(
        lambda x: x.fillna(x.median())
    )

print("NaN résiduels après imputation médiane :")
print((df[cols_mediane].isna().mean() * 100).round(1).to_string())

In [ ]:
df["nb_installations_5km"] = df["nb_installations_5km"].fillna(0)

In [ ]:
# Recalcul des lags après imputation de pm25_brute
df = add_lags(df)

In [ ]:
# Taux de valeurs manquantes par colonne
na_pct = df.isna().mean().sort_values(ascending=False)
na_pct = na_pct[na_pct > 0]

print("Colonnes avec valeurs manquantes :")
print((na_pct * 100).round(1).to_string())

# Visualisation
fig, ax = plt.subplots(figsize=(10, 5))
na_pct.plot(kind="bar", ax=ax)
ax.set_title("Taux de valeurs manquantes par colonne")
ax.set_ylabel("Taux (%)")
ax.set_xlabel("")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### 5.2 Suppression des lignes sans variable cible

Les lignes où `depasse_seuil_24h` est NaN correspondent aux dernières heures de chaque station (pas assez d'observations futures pour calculer la cible) et aux trous dans les séries temporelles. Elles ne peuvent pas être utilisées pour l'entraînement du modèle.

In [ ]:
n_avant = len(df)
df = df.dropna(subset=["depasse_seuil_24h"])
n_apres = len(df)
print(f"Lignes supprimées (cible NaN) : {n_avant - n_apres:,} ({(n_avant-n_apres)/n_avant:.1%})")
print(f"Shape final : {df.shape}")

In [ ]:
# Résumé final des NaN
na_final = df.isna().mean()
na_final = na_final[na_final > 0].sort_values(ascending=False)
print("NaN résiduels dans le DataFrame final :")
print((na_final * 100).round(1).to_string())

le reste des valeurs manqaantes sur les lages s'expliquent parce par exemple pour un lag de 24h les 24 premieres heures n'auront pas de valeurs.

## 6. Sauvegarde sur S3

In [ ]:
import io
import boto3
from dotenv import load_dotenv
load_dotenv()

s3 = boto3.client(
    "s3",
    endpoint_url          = os.getenv("S3_ENDPOINT_URL"),
    aws_access_key_id     = os.getenv("S3_ACCESS_KEY"),
    aws_secret_access_key = os.getenv("S3_SECRET_KEY"),
    aws_session_token     = os.getenv("S3_SESSION_TOKEN"),
)
BUCKET = os.getenv("S3_BUCKET")
KEY    = "projet-qualite-air/processed/dataset_consolide.csv"

# Sauvegarde en mémoire puis upload
buffer = io.StringIO()
df.to_csv(buffer, index=False)
buffer.seek(0)
s3.put_object(Bucket=BUCKET, Key=KEY, Body=buffer.getvalue())

print(f"✓ Dataset sauvegardé : s3://{BUCKET}/{KEY}")
print(f"  Lignes  : {len(df):,}")
print(f"  Colonnes: {len(df.columns)}")
print(f"  Taille  : {len(buffer.getvalue()) / 1e6:.1f} Mo")